# Import Utilities

In [ ]:
import sys
sys.path.append('../..')

from utils.llm_evaluation_utils import *
from utils.prompt_builder import build_rating_prompt, build_critique_prompt
from utils.data_setup import get_dataset_file_path, prepare_project_data
from utils.models_setup import setup_gemini, query_gemini_model
from utils.gemini_batch_manager import GeminiBatchManager

TASK_SUBSET = "1000"           # "all" | "1000" | "50"
PROMPTING_TYPE = "zero_shot"   # "zero_shot" | "few_shot"
STAGE = "critiques"              # or "critiques"
NUM_TRIALS = 1
MAX_TOKENS = 30000
TEMPERATURE = 1

# Schedules
rating_schedule_file_1000screens = Path("../OpenAI_Pipeline/rating_trial_schedule-1000_screens.parquet")
critiques_schedule_file_1000screens = Path("./OpenAI_Pipeline/critiques_trial_schedule-1000_screens.parquet")

if STAGE == "ratings":
    SYSTEM_MESSAGE = RATING_SYSTEM_MESSAGE
    build_prompt_fn = build_rating_prompt
    update_results_df_fn = update_rating_in_df
    schedule_file = rating_schedule_file_1000screens
elif STAGE == "critiques":
    SYSTEM_MESSAGE = CRITIQUES_SYSTEM_MESSAGE
    build_prompt_fn = build_critique_prompt
    update_results_df_fn = update_critiques_in_df
    schedule_file = critiques_schedule_file_1000screens
else:
    raise ValueError(f"Invalid stage: {STAGE}")

# === Model setup ===
client, cfg = setup_gemini(
    system_message=SYSTEM_MESSAGE,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS
    )

# Load Data

In [ ]:
SELECTED_TASKS = f"{TASK_SUBSET}_screens" if TASK_SUBSET != "all" else "all_tasks"
SELECTED_TASKS += f"-{PROMPTING_TYPE}"

# Batch folders (parameterized by SELECTED_TASKS)
batch_main_dir      = Path(f"./{STAGE}_Batching/Batch_Files-{SELECTED_TASKS}")
prompts_file  = batch_main_dir / f"prompts-{SELECTED_TASKS}.jsonl"

uicrit_file, base64_screens_file, few_shot_samples_file = get_dataset_file_path(STAGE)
uicrit_df = pd.read_parquet(uicrit_file)
base64_screens_df = pd.read_parquet(base64_screens_file)

if few_shot_samples_file:
    few_shot_samples_df = pd.read_parquet(few_shot_samples_file)
else:
    few_shot_samples_df = None

responses_df, results_paths = prepare_project_data(
    model_short=cfg["model_short"],
    num_trials=NUM_TRIALS,
    task_subset=TASK_SUBSET,
    shots=PROMPTING_TYPE,
    stage=STAGE
)
text_responses_jsonl_file = results_paths["results_jsonl"]
model_results_file        = results_paths["results_parquet"]

print("Responses Shape:", responses_df.shape)
responses_df.head(2)

Responses Shape: (997, 8)


,screen_task_id,screen_id,task,aesthetics_rating,learnability,efficiency,usability_rating,design_quality_rating
0,15_T01,15,Plan and Start Full Body Workouts,4.0,3.0,3.0,6.0,4.0
1,28_T01,28,Enter details to Sing In to Scotiabank.,8.0,5.0,5.0,10.0,8.0


## Defined Functions

In [4]:
# def prepare_gemini_errors_dict(dir, model_name):
#     '''Load and prepare a dictionary of IDs and question numbers with Blocked errors from a JSON file.'''
#     if '2.0' in model_name:
#         errors_file_name = 'ids_nums_with_Blocked_error-Gemini-2_0.json'
#     elif '2.5' in model_name:
#         errors_file_name = 'ids_nums_with_Blocked_error-Gemini-2_5.json'
        
#     file_path = os.path.join(dir, errors_file_name)
#     try:
#         with open(file_path, 'r') as f:
#             ids_questions_with_error = json.load(f)
#         # Convert the lists to sets
#         ids_questions_with_error = {key: set(value) for key, value in ids_questions_with_error.items()}
    
#     except FileNotFoundError:
#         ids_questions_with_error = {}
    
#     return ids_questions_with_error

# ids_questions_with_error = prepare_errors_dict('.', cfg["model_version"])


# **Batching**

In [5]:
batch_process = GeminiBatchManager(
    client=client,
    batch_main_dir=batch_main_dir, 
    selected_tasks=SELECTED_TASKS
    )

## Prepare Prompts JSONL File

In [ ]:
prompts_file = batch_process.generate_prompts_jsonl(
    schedule_file=rating_schedule_file_1000screens,
    responses_df=responses_df,
    screens_df=base64_screens_df,
    model_version=cfg["model_version"],
    system_message=SYSTEM_MESSAGE,
    guidelines=GUIDELINES,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
    prompting_type=PROMPTING_TYPE,
    samples_df=few_shot_samples_df,
    image_format="jpeg",
    build_prompt_fn=build_prompt_fn
    )

In [ ]:
# Check the first few lines of the generated JSONL file
with open(prompts_file, 'r', encoding='utf-8') as f:
    first = json.loads(f.readline())
    print(json.dumps(first, indent=2))
    # for _ in range(4):
    #     print(json.loads(f.readline()))

Since prompts in the JSONL file exceeds the maximum limit of tokens per day (900,000 TPD for tier 1), they have been splitted into multiple JSONL files, and then batched sequentially.

In [ ]:
batch_process.split_prompts_jsonl(max_lines_per_file=200-)    # Adjust as needed.

## **Batch Requests**

### Manually:

#### Step 2: Upload JSONL File

In [ ]:
# Upload JSONL file
batch_file_path = batch_process.prompts_dir / "prompts-batch_1.jsonl"

input_file_id = batch_process.upload_file(batch_file_path)

Display List of Files in OpenAI Account
This includes uploaded files, as well as output files generated by OpenAI

In [ ]:
_ = batch_process.list_uploaded_files()

Delete a File

In [ ]:
file_id_to_delete = 'files/XXXXXXXXXXXX'
                            

_ = batch_process.delete_file(file_id_to_delete)

#### Step 3: Create a Batch

In [15]:
# input_file_id = 'file-XXXXXXXXXXXXXXXXXXXXXXXX'

batch = batch_process.create_batch(model_version=cfg["model_version"])

✅ Batch created | ID: batches/6zmmw80hnhm70zxyei6zdzr1dy80y3sw1ooo | Status: JOB_STATE_PENDING


Display List of Batch Requests

In [ ]:
_ = batch_process.list_batches()

#### Step 4: Monitor and Retrieve Batch Information

In [ ]:
state, batch = batch_process.check_batch(verbose=True)

<u>Cancel</u> a Batch

In [5]:
batch_job_to_cancel = "batches/81xbbrkxbiaatwailih7np9ump25q1689gba"

_ = batch_process.cancel_batch(batch_job_to_cancel)

🚫 Cancelled


<u>Delete</u> a Batch

In [ ]:
batch_job_to_delete = "batches/XXXXXXXXXXXXXXXXXXXXXXXXXX"

_ = batch_process.delete_batch(batch_job_to_delete)

#### Step 5: Retrieve Response File Content

In [7]:
batch_id = "batches/6zmmw80hnhm70zxyei6zdzr1dy80y3sw1ooo"

results_file = batch_process.retrieve_batch_output(
    batch_id=batch_id,
    base_name="responses"
    )

✅ Saved batch output to responses.jsonl


### Alternatively, Steps 2-5: Batching sequentially automatically

In [ ]:
start_from_batch = 2
resume_batch = None

batch_process.process_batches_from(start_from_batch=start_from_batch, sleep_minutes=2, resume_batch_id=resume_batch)

## Extracting Results From Repsonses

In [ ]:
summary = batch_process.extract_results_from_responses(responses_df, update_results_df_fn)
print("Updated:", summary["updated"], "| Skipped:", len(summary["skipped"]), 
      "| Errors:", len(summary["errors"]))
# Inspect any problematic ids:
# print(summary["skipped"][:10], summary["errors"][:10])

responses_df.head(2)

Updated: 3001 | Skipped: 0 | Errors: 0


,screen_task_id,screen_id,task,aesthetics_rating,learnability,efficiency,usability_rating,design_quality_rating
0,15_T01,15,Plan and Start Full Body Workouts,4.0,3.0,3.0,6.0,4.0
1,28_T01,28,Enter details to Sing In to Scotiabank.,8.0,5.0,5.0,10.0,8.0


Combine all responses in one file

In [ ]:
# combine all responses in one file
responses_dir = batch_process.responses_dir
output_file = batch_main_dir / f"responses-{SELECTED_TASKS}.jsonl"

count = 0
with output_file.open("w", encoding="utf-8") as out_f:
    for p in sorted(responses_dir.iterdir()):
        if not p.is_file():
            continue
        if p.resolve() == output_file.resolve():
            continue
        try:
            text = p.read_text(encoding="utf-8", errors="ignore")
            if text:
                if not text.endswith("\n"):
                    text += "\n"
                out_f.write(text)
                count += 1
        except Exception as exc:
            print(f"Skipping {p.name}: {exc}")

print(f"Written {count} files to {output_file}")

# **One-by-One Prompt Request**

In [ ]:
query_args = dict(
    client=client,
    model_version=cfg["model_version"],
    config=cfg["config"],
    max_tokens=MAX_TOKENS,
    temperature=TEMPERATURE,
    system=SYSTEM_MESSAGE,
    image_format="jpeg",
)

run_llm_inference(
    responses_df=responses_df,
    base64_screens_df=base64_screens_df,
    few_shot_samples_df=few_shot_samples_df,
    evaluation_aspects=EVALUATION_MAIN_ASPECTS,
    build_prompt_fn=build_prompt_fn,
    query_fn=query_gemini_model,
    query_args=query_args,
    save_jsonl_fn=save_response_text,
    update_results_df_fn=update_rating_in_df,
    output_jsonl=text_responses_jsonl_file,
    guidelines=GUIDELINES,
    prompting_type=PROMPTING_TYPE,
    requests_per_minute=cfg["rpm"],
    stage=STAGE
)

In [ ]:
# # Calculate the delay based on your rate limit
# requests_limit_per_minute = 150
# base_delay = 60.0 / requests_limit_per_minute

# incomplete_rows = responses_df[responses_df[EVALUATION_MAIN_ASPECTS].isnull().any(axis=1)]

# for index, row in incomplet e_rows.iterrows():
#     screen_id = row['screen_id']
#     # skip if screen_id is in column screen_id in few_shot_samples_df 
#     if screen_id in few_shot_samples_df['screen_id'].values:
#         # drop that row from responses_df
#         responses_df = responses_df[responses_df['screen_id'] != screen_id]
#         continue
#     screen_task_id = row['screen_task_id']
#     base64_string = base64_screens_labeled_df.loc[screen_id, 'base64_screen']
#     # trial = row['trial']
#     # if trial == 1:
#     print(f"Processing: index={index} | screen_task_id={screen_task_id}")
    
#     for aspect in EVALUATION_MAIN_ASPECTS:
#         if pd.notnull(row[aspect]):  # Skip if already filled
#             continue
#         prompt = build_prompt_fn(row['task'], GUIDELINES, evaluate=aspect, prompting_type='few-shot', samples=few_shot_samples_df)
#         contents = create_few_shot_content()
                
#         try:
#             # response = model.generate_content(contents)
#             response = generate_content_with_backoff(client, prompt, base64_string, few_shot_samples_df, base_delay=base_delay)
#             if response:
#                 save_json_response(response.text, screen_id, screen_task_id, trial=None, metric=aspect, jsonl_responses_file=jsonl_responses_file)
#                 update_rating_in_df(responses_df, screen_task_id, trial=None, metric=aspect, response=response.text)
#             else:
#                 raise Exception("Could not update responses_df; response:", response)
            
#         except Exception as e:
#             print(f'Error with {screen_task_id}: {e}')
#             if 'candidate.safety_ratings' in str(e) or 'response.prompt_feedback' in str(e):    # BlockedPromptException
#                 if screen_task_id not in ids_questions_with_error:
#                     ids_questions_with_error[screen_task_id] = set()  # Initialize the set if it's the first occurrence of the screen_task_id
#                 # ids_questions_with_error[screen_task_id].add(trial)
#                 continue

#         time.sleep(base_delay)

# Explore Results

In [ ]:
# print('Number of errors: ', len(ids_questions_with_error))
# ids_questions_with_error

In [ ]:
columns_with_none = (responses_df.isna() | (responses_df == '')).sum()
columns_with_none

In [31]:
rows_with_none = responses_df[responses_df.isna().any(axis=1)]
rows_with_none

,screen_task_id,screen_id,task,aesthetics_rating,learnability,efficiency,usability_rating,design_quality_rating


## Store Errors to JSON File

In [ ]:
# def set_encoder(obj):
#     if isinstance(obj, set):
#         return list(obj)        # convert sets to lists
#     raise TypeError('Object of type set is not JSON serializable')

# # save ids and questions number that encountered errors
# errors_file_name = f"ids_nums_with_Blocked_error-{cfg["model_short"]}.json"
# with open(errors_file_name, 'w') as f:
#     json.dump(ids_questions_with_error, f, default=set_encoder)

# Store Results

In [32]:
responses_df.to_parquet(model_results_file, index=False)